# PushT visual Diffusion Policy on ManiSkill

Choose **Runtime > Change runtime type > T4 GPU** before running. This notebook targets Python 3.12 and delegates substantive code to the repository so experiments remain reproducible.

In [ ]:
import platform
import subprocess

print("Python:", platform.python_version())
subprocess.run(["nvidia-smi"], check=True)

Set `REPO_URL` to the GitHub repository you pushed from Cursor.

In [ ]:
REPO_URL = "https://github.com/YOUR_USER/YOUR_REPO.git"
assert "YOUR_USER" not in REPO_URL, "Edit REPO_URL first"
repo_name = REPO_URL.rstrip("/").rsplit("/", 1)[-1].removesuffix(".git")
!git clone {REPO_URL} /content/{repo_name}
%cd /content/{repo_name}/t-i
!bash scripts/setup_colab.sh

## Prepare the dataset
The raw download has no RGB observations. Replay 100 demonstrations to add them. `REPLAY_ENVS=64` is a Colab-conscious setting; 256 matches the upstream script more closely if VRAM permits.

In [ ]:
!NUM_DEMOS=100 REPLAY_ENVS=64 bash scripts/prepare_pusht_demos.sh

## Smoke test
Ten updates and one evaluation episode verify the complete data/model/simulator path. This is not meaningful training.

In [ ]:
!python train_dp.py --config configs/smoke.yaml

## Persistent full run
Save checkpoints to Drive so a Colab disconnect does not erase them.

In [ ]:
from google.colab import drive

drive.mount("/content/drive")
OUTPUT_DIR = "/content/drive/MyDrive/ripl-pusht-runs"

In [ ]:
!python train_dp.py --config configs/pusht_rgb.yaml --output-dir {OUTPUT_DIR}

After training, set `CHECKPOINT` to a `final.pt` or `best_success_once.pt` file. See README.md for resume and TensorBoard commands.

In [ ]:
CHECKPOINT = f"{OUTPUT_DIR}/RUN_NAME/checkpoints/final.pt"
assert "RUN_NAME" not in CHECKPOINT, "Set CHECKPOINT first"
!python eval_dp.py --checkpoint {CHECKPOINT} --num-eval-episodes 20 --num-eval-envs 10